# broadcast-source-fanout — ex1: fan a class-embedding table to a batch via labels

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `broadcast-source-fanout`. Running the final beacon cell reports progress against the `Generative: Broadcast source fan-out` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: Broadcast source fan-out` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`broadcast-source-fanout`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcast-source-fanout"
DD_SUBTOPIC = "Generative: Broadcast source fan-out"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Broadcast source fan-out — quick refresher

Class-conditional models hold ONE embedding per class — a `(num_classes, D)` parameter matrix. To turn that into a per-batch embedding, you index by labels:

```python
embed_table = nn.Parameter(t.randn(num_classes, D))
per_sample = embed_table[labels]   # labels: (B,) → per_sample: (B, D)
```

**This IS broadcasting.** A `(num_classes, D)` source tensor gets fanned out to a `(B, D)` tensor with possibly-repeated rows. Two labels that are equal get the same embedding row — that's what makes it *class-conditional*.

**Equivalent forms.** `embed_table[labels]` is identical to `F.embedding(labels, embed_table)` is identical to `nn.Embedding(num_classes, D)(labels)`. The Module form gets you `.weight` as a `nn.Parameter` automatically; the raw-index form is fine when you already have the table as a Parameter elsewhere.

**Gradient flow.** Gradients flow ONLY to the rows that were indexed — embedding lookup is differentiable in a sparse way. Classes absent from the batch receive zero gradient that step.

### Exercise 1 — fan a class-embedding table to a batch via labels

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply integer-label indexing `embed_table[labels]` to fan a `(num_classes, D)` class-conditional embedding table out to a `(B, D)` per-sample embedding tensor.
> Keywords: embedding, fanout, label-indexing, broadcast-source
> ```

**KCs targeted:** `embed-table-index-by-labels`, `fanout-shape-preserves-D`

Implement `ex1_fanout_class_embeddings(embed_table, labels)`. The class-conditional embedding lookup that conditional GANs and VAEs use to inject label information into the generator:

1. `embed_table` has shape `(num_classes, D)` — one row per class.
2. `labels` is a 1-D `int64` tensor of length `B`, each value in `[0, num_classes)`.
3. Return `embed_table[labels]` — shape `(B, D)`, where row `i` is `embed_table[labels[i]]`.
4. Two samples with the same label get the SAME embedding row (this is the 'broadcast' / fan-out semantics).

Input: `embed_table` `(num_classes, D)`, `labels` `(B,)` int64.
Output: `(B, D)` tensor of same dtype as `embed_table`.

The visualization renders the embedding table on the left and the per-sample fan-out result on the right, so you can see how each row of the output is copied from the table indexed by the label.

In [ ]:
def ex1_fanout_class_embeddings(embed_table: Tensor, labels: Tensor) -> Tensor:
    return embed_table[labels]


<details><summary>Solution</summary>

```python
def ex1_fanout_class_embeddings(embed_table: Tensor, labels: Tensor) -> Tensor:
    return embed_table[labels]
```

**Integer-index advanced indexing.** `embed_table[labels]` is exactly equivalent to `F.embedding(labels, embed_table)` — the Module form (`nn.Embedding`) is just a wrapper around this indexing op plus a `nn.Parameter`-wrapped weight matrix.

**Fan-out IS broadcasting.** A single embedding row `embed_table[c]` gets copied to every batch position whose label equals `c`. This is the same kind of source-fan-out as `x.expand(B, D)` — one source, many destinations — but driven by label values instead of a fixed shape rule.

**Gradient flow is sparse.** When `embed_table.requires_grad=True`, backward only updates the rows that were indexed. Classes absent from the batch receive zero gradient that step — which is why you see embedding tables sometimes drift slowly per row (rare classes are touched rarely).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()